# ReDB Benchmark Analysis

This notebook analyzes the CSV files produced by `benchmark/run_benchmark.py`.

The current input files are:

- `benchmark/results/all_raw.csv`: one row per query process.
- `benchmark/results/all_summary.csv`: one row per concurrency level.

This notebook filters to the latest run in the aggregate CSVs, so older synced runs are ignored.

The comments below explain what each table or plot means so the notebook can double as a short benchmark report.


In [ ]:
import pandas as pd
from pathlib import Path

results_dir = Path("results")
raw_all = pd.read_csv(results_dir / "all_raw.csv")
summary_all = pd.read_csv(results_dir / "all_summary.csv")

selected_run_id = "20260616T212422Z"
raw = raw_all[raw_all["run_id"] == selected_run_id].copy()
summary = summary_all[summary_all["run_id"] == selected_run_id].sort_values("concurrency")
measured = raw[(raw["status"] == "ok") & (raw["warmup"] == False)]

raw.shape, summary.shape, measured.shape


## Final read

This benchmark run completed successfully: **45 measured query processes**, **0 failures**, buffer size **20**, and `use_index = False`.

The useful headline is that concurrency improves throughput, but each individual query gets slower as workers compete for CPU, memory, and storage. Throughput rises from **0.117 qps** at concurrency 1 to **0.348 qps** at concurrency 4, which is a **2.98x speedup**. Ideal scaling would be 4.00x, so the engine benefits from parallel independent JVMs but is hitting shared-machine limits.

Latency moves the other way. Mean query latency increases from **8,480 ms** at concurrency 1 to **10,322 ms** at concurrency 4, a **21.7% increase**. That is expected for this benchmark design: ReDB is single-threaded per process, so concurrency comes from multiple JVMs fighting for the same EC2 resources.

Memory also scales almost linearly with active workers. Aggregate peak RSS grows from **544 MB** at concurrency 1 to **2,011 MB** at concurrency 4. The lowest sampled available host memory at concurrency 4 was **13,055 MB**, and peak sampled swap use was **0 MB**. If swap stays near zero, major faults are more likely file-backed page loads/cache misses than true RAM exhaustion.

The slowest workload is **`medium_t_range`**, averaging **13,714 ms** at concurrency 4. It also returns the most rows in this run: {'medium_t_range': 24225, 'small_a_range': 14525, 'medium_m_range': 11047}.


                ## Summary by concurrency

                This table is the main benchmark view. `throughput_qps` should go up with concurrency. `latency_mean_ms` and `latency_p95_ms` show the cost paid by each individual query as more workers run at the same time.

                | concurrency | throughput_qps | latency_mean_ms | latency_p50_ms | latency_p95_ms | makespan_seconds | cpu_utilization_cores | aggregate_peak_rss_mb | major_faults_total | host_cpu_count | host_memory_total_mb | host_memory_available_min_mb | host_swap_used_max_mb | host_cpu_utilization_mean_pct | host_cpu_utilization_max_pct | host_loadavg_1m_max | throughput_speedup_vs_c1 | latency_change_vs_c1_pct |
| ----------- | -------------- | --------------- | -------------- | -------------- | ---------------- | --------------------- | --------------------- | ------------------ | -------------- | -------------------- | ---------------------------- | --------------------- | ----------------------------- | ---------------------------- | ------------------- | ------------------------ | ------------------------ |
| 1           | 0.117          | 8480.0          | 7719.0         | 11543.0        | 128.4            | 1.15                  | 544.0                 | 0                  | 8.0            | 15699.4              | 14510.6                      | 0.0                   | 14.5                          | 55.0                         | 1.3                 | 1.0                      | 0.0                      |
| 2           | 0.209          | 9071.0          | 8306.0         | 12386.0        | 71.9             | 2.23                  | 998.0                 | 29                 | 8.0            | 15699.4              | 14057.5                      | 0.0                   | 28.3                          | 100.0                        | 1.9                 | 1.786                    | 7.0                      |
| 4           | 0.348          | 10322.0         | 9526.0         | 14322.0        | 43.1             | 4.22                  | 2011.0                | 130                | 8.0            | 15699.4              | 13054.5                      | 0.0                   | 53.2                          | 100.0                        | 3.5                 | 2.981                    | 21.7                     |


## Throughput

Throughput improves as concurrency increases, but it is sublinear. At concurrency 4, the run reaches about 3.0x the single-worker throughput, not 4.0x. That gap is the shared-resource overhead from CPU scheduling, JVM overhead, memory pressure, and disk/cache contention.

![Throughput plot](results/analysis/throughput.svg)


## Latency

Mean and p95 latency rise as concurrency increases. This is normal for the benchmark because it launches more independent JVMs on the same machine. The important question is whether the throughput gain is worth the extra per-query latency.

![Latency plot](results/analysis/latency.svg)


                ## Workload comparison

                The workloads return different numbers of rows, so they are not equally expensive. `medium_t_range` is consistently the slowest because it returns the largest result set in this run.

                | concurrency | workload       | runs | result_count | mean_query_ms | p95_query_ms | mean_wall_ms | mean_cpu_ms | mean_rss_mb |
| ----------- | -------------- | ---- | ------------ | ------------- | ------------ | ------------ | ----------- | ----------- |
| 1           | medium_m_range | 5    | 11047        | 6361.0        | 6455.0       | 6443.0       | 7722.0      | 485.0       |
| 1           | medium_t_range | 5    | 24225        | 11371.0       | 11530.0      | 11453.0      | 12786.0     | 510.0       |
| 1           | small_a_range  | 5    | 14525        | 7709.0        | 7870.0       | 7792.0       | 8976.0      | 510.4       |
| 2           | medium_m_range | 5    | 11047        | 6745.0        | 6870.0       | 6823.0       | 8354.0      | 451.9       |
| 2           | medium_t_range | 5    | 24225        | 12125.0       | 12360.0      | 12210.0      | 13848.0     | 500.7       |
| 2           | small_a_range  | 5    | 14525        | 8342.0        | 8547.0       | 8431.0       | 9924.0      | 458.6       |
| 4           | medium_m_range | 5    | 11047        | 7774.0        | 8028.0       | 7868.0       | 9520.0      | 502.5       |
| 4           | medium_t_range | 5    | 24225        | 13714.0       | 14304.0      | 13813.0      | 15760.0     | 529.7       |
| 4           | small_a_range  | 5    | 14525        | 9477.0        | 9847.0       | 9571.0       | 11070.0     | 495.6       |

                ![Workload latency plot](results/analysis/workload_latency.svg)


                ## Resource use

                Aggregate RSS rises with concurrency because each worker is its own JVM. CPU utilization also rises with concurrency, which means the EC2 machine is actually doing more work in parallel rather than just queueing everything serially.

                On this run, the host-level fields show the machine was not close to exhausting memory: available memory stayed far above the workers' aggregate RSS and swap stayed at 0 MB. That means the major faults should be interpreted as ordinary disk-backed page loads/cache misses, not evidence that the EC2 instance ran out of RAM.

                ![Host memory plot](results/analysis/host_memory.svg)

![Host CPU plot](results/analysis/host_cpu.svg)



In [ ]:
# Useful follow-up query: compare only successful measured rows.
measured.groupby(["concurrency", "workload"]).agg(
    runs=("query_elapsed_ms", "size"),
    result_count=("result_count", "first"),
    mean_query_ms=("query_elapsed_ms", "mean"),
    p95_query_ms=("query_elapsed_ms", lambda s: s.quantile(0.95)),
    mean_rss_mb=("peak_rss_bytes", lambda s: s.mean() / 1024 / 1024),
).round(2)


## What to run next

For a stronger report, run the same benchmark with `--index` and compare `use_index = true` vs `false`. Also try a larger buffer size, for example 50 or 100, to see whether the BNL joins are buffer-limited.

Suggested next labels:

- `no-index-buffer-20`
- `index-buffer-20`
- `no-index-buffer-50`
- `index-buffer-50`
